In [8]:
# === CELL 1: IMPORTS AND CONFIGURATION ===
import requests
import pandas as pd
import json
import time
import random
import os
import re
from datetime import datetime
import datetime as dt

# ── ID Continuity ──────────────────────────────────────────────────────────
existing_max_id = 0
for csv_path in ["data/consumer_complaints.csv", "data/indiankanoon_complaints.csv", "data/medianama_complaints.csv"]:
    if os.path.exists(csv_path):
        df_check = pd.read_csv(csv_path)
        if "Unique ID" in df_check.columns:
            ids = df_check["Unique ID"].dropna().str.extract(r'(\d+)')[0].astype(float)
            if not ids.empty:
                existing_max_id = max(existing_max_id, int(ids.max()))
next_id = existing_max_id + 1
print(f"Starting Unique ID from: NA-{next_id:04d}")

# ── Paths & APIs ───────────────────────────────────────────────────────────
PULLPUSH_URL = "https://api.pullpush.io/reddit/search/submission/"
COMMENTS_URL = "https://api.pullpush.io/reddit/search/comment/"

TARGET_SUBREDDITS = [
    "india", "LegalAdviceIndia", "IndiaInvestments",
    "bangalore", "mumbai", "delhi", "hyderabad",
    "chennai", "pune", "PersonalFinanceIndia", "scams"
]

OUTPUT_CSV = "data/reddit_complaints.csv"
OUTPUT_JSON = "data/reddit.json"
OUTPUT_EXCEL = "data/reddit_complaints.xlsx"
TXT_DIR = "data/txt"

os.makedirs(TXT_DIR, exist_ok=True)
os.makedirs("data", exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# Date range for research: Jan 2014 to Dec 2025
# PullPush uses Unix timestamps
START_TIMESTAMP = int(dt.datetime(2014, 1, 1).timestamp())
END_TIMESTAMP = int(dt.datetime(2025, 12, 31).timestamp())

Starting Unique ID from: NA-15686


In [3]:
# === CELL 2: FULL KEYWORD TAXONOMY ===
KEYWORD_TAXONOMY = {
    "General Cybercrime / Cyber Fraud Terms": {
        "General Cybercrime": ["cyber crime", "cybercrime", "cyber fraud", "online fraud", "internet fraud", "digital fraud"],
        "Cyber Scam": ["cyber scam", "online scam", "internet scam", "online cheating"],
        "Financial Cyber Fraud": ["financial fraud online", "net banking fraud", "e-banking fraud"]
    },
    "UPI and Digital Payment Fraud": {
        "UPI Fraud": ["UPI fraud", "UPI scam", "Google Pay fraud", "PhonePe fraud", "Paytm fraud", "BHIM fraud"],
        "QR Code Fraud": ["QR code scam", "QR code fraud", "scan and pay fraud"],
        "Payment Link Fraud": ["payment link fraud", "collect request scam", "fake payment link"],
        "Mobile Wallet Fraud": ["mobile wallet fraud", "e-wallet scam", "digital wallet fraud"]
    },
    "OTP and Authentication Fraud": {
        "OTP Fraud": ["OTP fraud", "OTP scam", "OTP theft"],
        "SIM Swap": ["SIM swap fraud", "SIM cloning", "duplicate SIM fraud"],
        "KYC Fraud": ["KYC fraud", "KYC scam", "Aadhaar KYC scam"]
    },
    "Digital Arrest Scam": {
        "Digital Arrest": ["digital arrest", "digital arrest scam", "fake arrest", "video call arrest"],
        "Impersonation Scam": ["police impersonation scam", "CBI fraud call", "customs fraud call", "TRAI scam call", "ED scam call"],
        "Video Call Coercion": ["video call scam", "video call blackmail", "video call extortion", "fake interrogation"]
    },
    "Phishing, Vishing, and Smishing": {
        "Phishing": ["phishing", "phishing attack", "phishing email", "fake website", "spoof website"],
        "Vishing": ["vishing", "voice phishing", "fraud call", "fake bank call"],
        "Smishing": ["smishing", "SMS fraud", "SMS scam", "phishing SMS"]
    },
    "Online Lending and Loan App Fraud": {
        "Loan App Fraud": ["loan app fraud", "instant loan scam", "loan app harassment", "illegal loan app"],
        "Loan App Extortion": ["loan app blackmail", "loan app threat", "morphed photos loan", "recovery agent threat"]
    },
    "Investment and Trading Fraud": {
        "Investment Scam": ["investment scam", "Ponzi scheme", "online investment fraud", "crypto scam", "bitcoin fraud", "forex trading scam", "pig butchering"],
        "Stock Market Fraud": ["stock market scam", "share trading fraud", "demat fraud", "pump and dump"],
        "Task Scam": ["task fraud", "part time job scam", "work from home scam", "Telegram task scam"]
    },
    "Identity Theft and Data Breach": {
        "Identity Theft": ["identity theft", "identity fraud", "Aadhaar misuse", "PAN fraud"],
        "Data Breach": ["data breach", "data leak", "data theft", "customer data breach"]
    },
    "Social Engineering and Romance/Sextortion": {
        "Social Engineering": ["social engineering fraud", "manipulation scam", "trust scam"],
        "Romance Scam": ["romance scam", "dating fraud", "matrimonial fraud", "honey trap", "catfishing fraud"],
        "Sextortion": ["sextortion", "webcam blackmail", "nude video blackmail"]
    },
    "E-Commerce and Delivery Fraud": {
        "E-Commerce Fraud": ["e-commerce fraud", "online shopping fraud", "fake product scam", "Flipkart fraud", "Amazon fraud", "refund scam"],
        "Delivery Fraud": ["fake delivery", "courier fraud", "customs duty scam", "parcel scam", "delivery OTP scam"]
    },
    "Ransomware and Malware": {
        "Ransomware": ["ransomware attack", "cyber ransom", "data encryption attack"],
        "Banking Malware": ["banking trojan", "banking malware", "keylogger fraud", "AnyDesk fraud", "TeamViewer scam", "screen sharing scam"]
    },
    "Emerging and Miscellaneous Fraud Types": {
        "Deepfake Fraud": ["deepfake scam", "deepfake fraud", "AI voice scam", "voice cloning scam"],
        "Utility Scam": ["electricity bill scam", "utility fraud", "disconnection scam"],
        "Aadhaar Fraud": ["Aadhaar fraud", "Aadhaar scam", "biometric fraud", "AEPS fraud"],
        "Cyber Stalking": ["cyber stalking", "cyber bullying", "online harassment", "digital harassment"]
    }
}

In [4]:
# === CELL 3: HELPER FUNCTIONS ===
def classify_narrative_type(text):
    text_lower = text.lower()
    if any(x in text_lower for x in ["i lost", "i was scammed", "they took", "money deducted",
                                       "i got cheated", "duped", "fell for", "lost money",
                                       "my account", "amount debited"]):
        return "VICTIM"
    elif any(x in text_lower for x in ["almost", "tried to scam", "i did not", "i refused",
                                         "i avoided", "beware", "warning", "how i avoided",
                                         "did not share", "suspicious"]):
        return "NEAR-MISS"
    return "THIRD-PARTY"

def clean_text(text):
    if not text:
        return ""
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def safe_save(df, csv_path, json_path, json_data):
    """Save via temp file to avoid PermissionError if Excel has file open."""
    try:
        temp_csv = csv_path + ".tmp"
        df.to_csv(temp_csv, index=False, encoding="utf-8-sig")
        os.replace(temp_csv, csv_path)
        temp_json = json_path + ".tmp"
        with open(temp_json, "w", encoding="utf-8") as f:
            json.dump(json_data, f, indent=4, ensure_ascii=False)
        os.replace(temp_json, json_path)
        print(f"  ✅ Checkpoint saved: {len(df)} records")
    except Exception as e:
        print(f"  ⚠️ Checkpoint save failed (data still in memory): {e}")

In [5]:
# === CELL 4: PULLPUSH FETCH FUNCTION ===
def fetch_reddit_posts(keyword, subreddit, before=None, size=100):
    """Fetch posts from PullPush API for a keyword in a subreddit."""
    params = {
        "q": keyword,
        "subreddit": subreddit,
        "size": size,
        "after": START_TIMESTAMP,
        "sort": "desc",
        "sort_type": "created_utc"
    }
    if before:
        params["before"] = before

    try:
        time.sleep(random.uniform(1.0, 3.0))
        response = requests.get(PULLPUSH_URL, params=params, headers=HEADERS, timeout=15)
        if response.status_code != 200:
            print(f"    Failed: status {response.status_code}")
            return []
        data = response.json()
        posts = data.get("data", [])
        return posts
    except Exception as e:
        print(f"    Error fetching '{keyword}' from r/{subreddit}: {e}")
        return []

In [9]:
# === CELL 5: MAIN SCRAPING LOOP ===
# Setup duplicates loading
existing_post_ids = set()
if os.path.exists(OUTPUT_JSON):
    with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
        master_data_json = json.load(f)
    for item in master_data_json:
        if item.get("Reddit Post ID"):
            existing_post_ids.add(item["Reddit Post ID"])
        elif item.get("Notes") and "Reddit ID: " in item["Notes"]:
            existing_post_ids.add(item["Notes"].split("Reddit ID: ")[-1].split(" |")[0])
else:
    master_data_json = []
print(f"Already scraped: {len(existing_post_ids)} posts")

new_results_count = 0
today_date = datetime.now().strftime("%Y-%m-%d")

for subreddit in TARGET_SUBREDDITS:
    print(f"\n=== Scraping Subreddit: r/{subreddit} ===")
    
    for parent_category, subcategories in KEYWORD_TAXONOMY.items():
        for subcat, keyword_list in subcategories.items():
            for keyword in keyword_list:
                print(f"  Keyword: '{keyword}'")
                before = END_TIMESTAMP
                
                while True:
                    posts = fetch_reddit_posts(keyword, subreddit, before=before, size=100)
                    
                    if not posts:
                        break
                        
                    batch_valid = 0
                    for post in posts:
                        post_id = post.get('id', '')
                        if not post_id or post_id in existing_post_ids:
                            continue
                        existing_post_ids.add(post_id)
                            
                        created_utc = post.get('created_utc')
                        if created_utc is None:
                            continue
                        try:
                            created_utc = int(float(str(created_utc)))
                        except (ValueError, TypeError):
                            continue
                        if created_utc < START_TIMESTAMP or created_utc > END_TIMESTAMP:
                            continue
                            
                        readable_date = datetime.fromtimestamp(created_utc).strftime("%Y-%m-%d")
                        
                        selftext = post.get('selftext', '').strip()

                        # Handle deleted, removed, or empty posts
                        if not selftext or selftext in ['[deleted]', '[removed]']:
                            post_body = f"[No body text — title only]\n\nTitle contains full narrative:\n{post.get('title', '')}"
                        else:
                            post_body = clean_text(selftext)
                            
                        title = post.get('title', '')
                        permalink = post.get('permalink', '')
                        score = post.get('score', 0)
                        author = post.get('author', 'unknown')
                        
                        existing_max_id += 1
                        assigned_id = f"NA-{existing_max_id:04d}"
                        txt_filename = f"{assigned_id}.txt"
                        
                        # Build txt file content
                        txt_content = f"SUBREDDIT: r/{post.get('subreddit', '')}\n"
                        txt_content += f"TITLE: {post.get('title', '')}\n"
                        txt_content += f"DATE: {readable_date}\n"
                        txt_content += f"SCORE: {post.get('score', 0)}\n"
                        txt_content += f"URL: https://www.reddit.com{post.get('permalink', '')}\n\n"
                        txt_content += "--- POST TEXT ---\n\n"
                        txt_content += post_body

                        # Use title + post body combined for narrative classification
                        full_content = post.get('title', '') + ' ' + post_body
                        narrative_type = classify_narrative_type(full_content)
                        
                        with open(os.path.join(TXT_DIR, txt_filename), "w", encoding="utf-8") as f_txt:
                            f_txt.write(txt_content)
                            
                        record = {
                            "Reddit Post ID": post_id,
                            "Unique ID": assigned_id,
                            "Date of Collection": today_date,
                            "Collector Name": "Soubhik Sarkar",
                            "Source Platform": "Reddit",
                            "Source Publication": f"r/{post.get('subreddit')}",
                            "Original Date": readable_date,
                            "Title/Headline": title,
                            "URL": "https://www.reddit.com" + permalink,
                            "Search Query Used": keyword,
                            "Fraud Category": parent_category,
                            "Fraud Subcategory": subcat,
                            "Narrative Type": narrative_type,
                            "TXT File Name": txt_filename,
                            "Notes": f"Reddit ID: {post_id} | Score: {score} | Author: {author}"
                        }
                        
                        master_data_json.append(record)
                        new_results_count += 1
                        batch_valid += 1
                        print(f"    ✅ Saved {assigned_id}: {title[:40]}...")
                        
                        if new_results_count % 50 == 0:
                            df_temp = pd.DataFrame(master_data_json)
                            safe_save(df_temp, OUTPUT_CSV, OUTPUT_JSON, master_data_json)
                            
                    if len(posts) < 10 or not batch_valid:
                        break
                        
                    valid_times = []
                    for p in posts:
                        c_utc = p.get('created_utc')
                        if c_utc is not None:
                            try:
                                valid_times.append(int(float(str(c_utc))))
                            except (ValueError, TypeError):
                                pass
                    oldest_in_batch = min(valid_times) if valid_times else before
                    
                    if oldest_in_batch < START_TIMESTAMP:
                        break
                    before = oldest_in_batch

print(f"\nTotal new posts scraped this session: {new_results_count}")

Loaded 300 existing scraped posts from JSON.

=== Scraping Subreddit: r/india ===
  Keyword: 'cyber crime'
  Keyword: 'cybercrime'
    ✅ Saved NA-15686: [URGENT] I need assistance in identifyin...
    ✅ Saved NA-15687: I reported a crime but feel mentally str...
    ✅ Saved NA-15688: Can businesses in India asking for phone...
    ✅ Saved NA-15689: Cybercrime department flagged me for no ...
    ✅ Saved NA-15690: Can We Please Get Rid of the Cybercrime ...
    ✅ Saved NA-15691: hello everyone! i'm collecting primary d...
    ✅ Saved NA-15692: My pathetic experience with Airbnb booki...
    ✅ Saved NA-15693: New Scam in Town...
    ✅ Saved NA-15694: Cybercrime incompetency...
    ✅ Saved NA-15695: Phishing Scam – Entered Friend’s Card PI...
    ✅ Saved NA-15696: Students share their screen and play por...
    ✅ Saved NA-15697: Some Civic Sense for Us...
    ✅ Saved NA-15698: Cybercrime report...
    ✅ Saved NA-15699: Shocking Cybercrime Stats in India...
    ✅ Saved NA-15700: Ia there n

In [ ]:
# === CELL 6: FINAL SAVE ===
if new_results_count > 0:
    df_final = pd.DataFrame(master_data_json)
    safe_save(df_final, OUTPUT_CSV, OUTPUT_JSON, master_data_json)
    try:
        df_final.to_excel(OUTPUT_EXCEL, index=False)
        print(f"  ✅ Saved Excel to {OUTPUT_EXCEL}")
    except Exception as e:
        print(f"  ⚠️ Could not save Excel: {e}")
    print(f"✅ Final save complete. Total records: {len(df_final)}")
    display(df_final.tail(3))
else:
    print("No new data to save.")

In [6]:
# === CELL 7: DIAGNOSTIC TEST CELL ===
# Test one keyword in one subreddit
test_posts = fetch_reddit_posts("UPI fraud", "india", size=3)
print(f"Found {len(test_posts)} posts")
if test_posts:
    p = test_posts[0]
    print(f"Title: {p.get('title')}")
    print(f"Date: {datetime.fromtimestamp(p['created_utc']).strftime('%Y-%m-%d')}")
    print(f"Subreddit: r/{p.get('subreddit')}")
    print(f"Text preview: {p.get('selftext', '')[:300]}")
    print(f"URL: https://www.reddit.com{p.get('permalink')}")

Found 3 posts
Title: Zepto Delivery Agent Misused OTP — Product Not Delivered, ₹500 Free Cash Lost — Beware!
Date: 2025-04-28
Subreddit: r/india
Text preview: Sharing a serious incident that happened recently with Zepto, so others don’t fall for the same trap.

I ordered a **boAt Wave Aura Smartwatch** on Zepto for just **₹490**, using my **₹500 Free Cash**. Normally, this smartwatch costs around **₹900**, so it was a great deal. I chose **Cash on Deliver
URL: https://www.reddit.com/r/india/comments/1k9tgsj/zepto_delivery_agent_misused_otp_product_not/


In [ ]:
# === STANDALONE CLEANUP CELL ===
# Run this after scraping finishes to clean the dataset
import pandas as pd
import os
import json
import re

OUTPUT_CSV = "data/reddit_complaints.csv"
OUTPUT_JSON = "data/reddit.json"
OUTPUT_EXCEL = "data/reddit_complaints.xlsx"
TXT_DIR = "data/txt"

INDIA_KEYWORDS = ["india", "indian", "rupee", "inr", "₹", "upi", "paytm", "phonepe", "gpay", "bhim", "aadhaar", "neft", "imps", "rupees"]
# We use regex to match whole words or the rupee symbol individually
keyword_pattern = re.compile(
    r'\b(?:' + '|'.join([re.escape(k) for k in ["india", "indian", "rupee", "inr", "upi", "paytm", "phonepe", "gpay", "bhim", "aadhaar", "neft", "imps", "rupees"]]) + r')\b|₹',
    re.IGNORECASE
)

if os.path.exists(OUTPUT_CSV):
    df = pd.read_csv(OUTPUT_CSV)
    total_before = len(df)
    
    # Track indices for removed non-India posts
    indices_to_drop_scams = []
    
    print("Step 1: Filtering non-India posts from r/scams...")
    for idx, row in df.iterrows():
        if row.get("Source Publication", "") == "r/scams":
            title = str(row.get("Title/Headline", ""))
            txt_filename = row.get("TXT File Name", "")
            
            txt_content = ""
            if pd.notna(txt_filename) and txt_filename:
                txt_path = os.path.join(TXT_DIR, str(txt_filename))
                if os.path.exists(txt_path):
                    with open(txt_path, 'r', encoding='utf-8') as f:
                        txt_content = f.read()
                        
            combined_text = title + " " + txt_content
            if not keyword_pattern.search(combined_text):
                indices_to_drop_scams.append(idx)
                
    removed_scams_count = len(indices_to_drop_scams)
    df_dropped_scams = df.drop(index=indices_to_drop_scams)
    
    print("Step 2: Removing duplicates...")
    len_before_dedups = len(df_dropped_scams)
    df_dedup = df_dropped_scams.drop_duplicates(subset=["URL"], keep="first")
    df_dedup = df_dedup.drop_duplicates(subset=["TXT File Name"], keep="first")
    duplicates_removed = len_before_dedups - len(df_dedup)
    
    df_final = df_dedup.reset_index(drop=True)
    total_after = len(df_final)
    
    print("Step 3: Deleting orphaned TXT files...")
    original_txts = set(df["TXT File Name"].dropna().tolist())
    final_txts = set(df_final["TXT File Name"].dropna().tolist())
    removed_txts = original_txts - final_txts
    
    deleted_txt_count = 0
    # Safe deletion: verify the deleted txt belonged to Reddit
    for idx, row in df.iterrows():
        txt_file = row.get("TXT File Name")
        url = str(row.get("URL", ""))
        
        if txt_file in removed_txts and "reddit.com" in url:
            txt_path = os.path.join(TXT_DIR, str(txt_file))
            if os.path.exists(txt_path):
                os.remove(txt_path)
                deleted_txt_count += 1
            # Remove from target set
            removed_txts.remove(txt_file)

    print("\nStep 4: Cleanup Summary")
    print(f"  - Total records before cleanup: {total_before}")
    print(f"  - Records removed from r/scams (non-India): {removed_scams_count}")
    print(f"  - Duplicate records removed: {duplicates_removed}")
    print(f"  - Final record count: {total_after}")
    print(f"  - TXT files deleted: {deleted_txt_count}")
    print("\n  - Breakdown by Subreddit:")
    if "Source Publication" in df_final.columns:
        print(df_final["Source Publication"].value_counts().to_string())
        
    print("\nStep 5: Saving cleaned data safely...")
    temp_csv = OUTPUT_CSV + ".tmp"
    df_final.to_csv(temp_csv, index=False, encoding="utf-8-sig")
    os.replace(temp_csv, OUTPUT_CSV)
    
    temp_excel = OUTPUT_EXCEL + ".tmp"
    try:
        df_final.to_excel(temp_excel, index=False)
        os.replace(temp_excel, OUTPUT_EXCEL)
    except Exception as e:
        print(f"  ⚠️ Could not save Excel: {e}")
        
    if os.path.exists(OUTPUT_JSON):
        with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
            
        final_urls = set(df_final["URL"].dropna().tolist())
        cleaned_json = [item for item in json_data if item.get("URL") in final_urls]
        
        temp_json = OUTPUT_JSON + ".tmp"
        with open(temp_json, 'w', encoding='utf-8') as f:
            json.dump(cleaned_json, f, indent=4, ensure_ascii=False)
        os.replace(temp_json, OUTPUT_JSON)
        print(f"  - JSON reduced from {len(json_data)} to {len(cleaned_json)} items.")
        
    print("\n✅ Cleanup fully complete!")
else:
    print("❌ reddit_complaints.csv not found.")